# LC–MS metabolomikai pipeline (v2 mintára)

**ST000816** lipid peak mátrix: neg + pos ionmód, minták × jellemzők.

Lépések (klasszikus / NMR `Metabolomika_Pipeline_v2` szerkezethez hasonlóan): **adat → QC → előfeldolgozás → PCA → univariáns + vulkán → heatmap → PLS-DA**.

A függvények a `src/metabolomics_*.py` modulokban vannak.


## 0. Környezet és betöltés


In [ ]:
from pathlib import Path
import sys

# Projekt gyökér (Colab: állítsd a repo útvonalát)
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").is_dir() and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.download_dataset import download_dataset, combine_pos_neg_to_feature_matrix

data = download_dataset()
X_raw, meta = combine_pos_neg_to_feature_matrix(data)
print(X_raw.shape, meta.shape)
print(meta.head())


## 1. QC (mintánkénti összjel és hiányzás)


In [ ]:
from src.metabolomics_qc import qc_sample_table
from src.metabolomics_plotting import plot_qc_bars

qc = qc_sample_table(X_raw)
print(qc.describe())
fig = plot_qc_bars(qc, meta, hue_col="progressor_status")
fig.savefig("qc_overview.png", dpi=150, bbox_inches="tight")


## 2. Előfeldolgozás (medián norm, imputáció, log1p, Pareto)


In [ ]:
from src.metabolomics_preprocessing import preprocess_feature_matrix

X_proc, steps = preprocess_feature_matrix(X_raw)
print("Lépések:", " → ".join(steps))
X_proc.head()


## 3. Exploratív PCA


In [ ]:
from src.metabolomics_multivariate import fit_pca
from src.metabolomics_plotting import plot_pca_scores

pca_out = fit_pca(X_proc, n_components=5)
scores = pca_out["scores"]
var = pca_out["explained_variance_ratio"]
print("Magyarázott variancia (első 5 PC):", [round(float(v), 4) for v in var])

fig = plot_pca_scores(
    scores, meta,
    pc_x="PC1", pc_y="PC2",
    hue_col="progressor_status",
    explained=var,
    title="PCA — előfeldolgozott lipid mátrix",
)
fig.savefig("pca_scores.png", dpi=150, bbox_inches="tight")


## 4. Univariáns elemzés + vulkán (Baseline: Progressor vs Non-progressor)


In [ ]:
# Csak baseline látogatás
bl = meta["visit"] == "Baseline"
Xb = X_proc.loc[bl]
metab = meta.loc[bl]

from src.metabolomics_univariate import differential_analysis
from src.metabolomics_plotting import plot_volcano

diff = differential_analysis(
    Xb,
    metab["progressor_status"],
    group_a="Progressor",
    group_b="Non-progressor",
)
print(diff.head(15))
fig = plot_volcano(diff, alpha=0.05, fc_thresh=0.5)
fig.savefig("volcano_baseline_progressor.png", dpi=150, bbox_inches="tight")


## 5. Top eltérések heatmap


In [ ]:
from src.metabolomics_plotting import plot_top_features_heatmap

top_n = 25
top_feats = diff.nsmallest(top_n, "padj")["feature"].tolist()
fig = plot_top_features_heatmap(
    Xb,
    top_feats,
    metab,
    group_col="progressor_status",
    max_samples=50,
)
fig.savefig("heatmap_top_features.png", dpi=150, bbox_inches="tight")


## 6. PLS-DA (két osztály, score plot)


In [ ]:
from src.metabolomics_multivariate import fit_pls_binary
import matplotlib.pyplot as plt

pls = fit_pls_binary(Xb, metab["progressor_status"], positive_class="Progressor", n_components=2)
s_pls = pls["scores"]

fig, ax = plt.subplots(figsize=(7, 5))
for lab in metab["progressor_status"].unique():
    m = metab["progressor_status"] == lab
    ax.scatter(s_pls.loc[m, "LV1"], s_pls.loc[m, "LV2"], label=str(lab), s=45, alpha=0.85, edgecolors="white", linewidths=0.3)
ax.set_xlabel("LV1")
ax.set_ylabel("LV2")
ax.set_title("PLS (bináris Progressor címke)")
ax.legend(title="progressor_status")
ax.axhline(0, color="gray", lw=0.4)
ax.axvline(0, color="gray", lw=0.4)
fig.tight_layout()
fig.savefig("pls_scores.png", dpi=150, bbox_inches="tight")


## 7. Rövid értelmezés

- **QC**: extrém `total_signal` vagy `missing_frac` minták kiszűrhetők.
- **PCA**: fő irányok a legnagyobb közös variancia mentén; színezés a klinikai címkével.
- **Vulkán + FDR**: egyszerre nézünk hatást (log2FC) és bizonytalanságot (p / FDR).
- **PLS**: a címkéhez legjobban illeszkedő lineáris kombinációk (LC–MS adatra; nem egyezik meg az OPLS-DA összes formalizmusával, de a score plot hasonló szerepű).

A `NMR_Metabolomika_Pipeline_v2.ipynb` fájl nem volt a repositoryban; ez a notebook ugyanazt a **logikai sorrendet** követi, LC–MS lipid adatra szabva.
